In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

/var/folders/hy/r8vny_js1v3_fr88p6zjjgm80000gn/T/ipykernel_37713/3267508256.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
loader = PyPDFLoader("../data/SuyashRaj_082026 2.pdf")
docs = loader.load()
len(docs)

1

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)
len(split_docs)

4

In [9]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [10]:
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings 
)

In [11]:
query = "What is whole summary of the documents?"
data = vector_store.similarity_search(query)

In [12]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2)

In [13]:
def get_context(query:str):
    data = vector_store.similarity_search(query = query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
    return {
        "context": context,
        "question": query
        
    }

In [14]:
prompt = PromptTemplate.from_template(
    """You are a helpful assistant that answers questions based on the context provided.
    "If the answer is not in the context, say 'I don't know'
    Context: {context}
    Question: {question}""")

In [15]:
rag_chain = get_context | prompt | llm

In [17]:
res = rag_chain.invoke("What is the summary of the resume" \
"")

/Users/suyashraj/.gemini/antigravity/scratch/data-science-portfolio/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [18]:
from IPython.display import Markdown, display

text = res.content[0]["text"]

display(Markdown(text))

Based on the provided resume, here is a summary of **Suyash Raj**:

* **Education:** Holds a B.Tech degree from Netaji Subhash University Of Technology (September 2020 – July 2024).
* **Current Role & Experience:** 
  * **Backend Development Engineer at Tata Consultancy Services** (*Feb 2025 – Present*): Focuses on building scalable Python backend services, REST APIs (FastAPI, Flask), Apache Kafka data pipelines, ETL workflows, and RAG-based retrieval systems for real-time platforms.
  * **Software Development Engineer Intern at Courpedia** (*Aug 2024 – Dec 2024*): Developed full-stack applications and REST APIs using Node.js, Express.js, and MongoDB.
* **Technical Skills:**
  * **Languages:** Python, Java, JavaScript, SQL
  * **Backend & Systems:** FastAPI, Flask, Node.js, Django, Microservices, Kafka, REST APIs, Distributed Systems
  * **AI/ML:** RAG, Langchain, HuggingFace, Ollama, Scikit-learn, Prompt Engineering
  * **Databases & Cloud:** PostgreSQL, MySQL, MongoDB, Redis, Elasticsearch, OpenSearch, AWS, Azure, Docker, Git, CI/CD
* **Projects:**
  * **OptiStack AI Backend Performance Optimizer & RAG Platform:** Created a RAG incident assistant using Qdrant vector DB and SentenceTransformers, alongside an ML model using Scikit-Learn for anomaly detection.
  * **Movie Recommendation System:** Built an end-to-end recommendation system using Python, Scikit-learn, Pandas, FastAPI, and AWS with NLP feature extraction.